In [1]:
# ============================================
# 02 - CONVERSATION RECONSTRUCTION
# ============================================

import pandas as pd
from pathlib import Path

RAW_FILE = Path("../data/raw/twcs.csv")

print("Starting conversation reconstruction...")

Starting conversation reconstruction...


In [2]:
# ============================================
# 02 - CONVERSATION RECONSTRUCTION
# Step 2.1 — Load data
# ============================================

import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path("../data/raw/twcs.csv")

USECOLS = [
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id"
]

df = pd.read_csv(
    RAW_FILE,
    usecols=USECOLS,
    nrows=100_000
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (100000, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_FILE = Path("../data/raw/twcs.csv")

USECOLS = [
    "tweet_id",
    "author_id",
    "inbound",
    "created_at",
    "text",
    "response_tweet_id",
    "in_response_to_tweet_id"
]

df = pd.read_csv(
    RAW_FILE,
    usecols=USECOLS,
    nrows=100_000
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Shape: (100000, 7)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0


In [5]:
# ============================================
# Cell 2 — Normalize IDs and timestamps
# ============================================

df["tweet_id"] = df["tweet_id"].astype(str).str.strip()

df["author_id"] = (
    df["author_id"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["in_response_to_tweet_id"] = (
    df["in_response_to_tweet_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("nan", "")
)

df["response_tweet_id"] = (
    df["response_tweet_id"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("nan", "")
)

df["created_at"] = pd.to_datetime(
    df["created_at"],
    errors="coerce"
)

print("Data types after normalization:\n")
print(df.dtypes)

print("\nMissing values:\n")
print(df.isna().sum())

C:\Users\SHUBHANSHI GUPTA\AppData\Local\Temp\ipykernel_19640\152449431.py:30: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["created_at"] = pd.to_datetime(


Data types after normalization:

tweet_id                                   str
author_id                                  str
inbound                                   bool
created_at                 datetime64[us, UTC]
text                                       str
response_tweet_id                          str
in_response_to_tweet_id                    str
dtype: object

Missing values:

tweet_id                   0
author_id                  0
inbound                    0
created_at                 0
text                       0
response_tweet_id          0
in_response_to_tweet_id    0
dtype: int64


In [6]:
# ============================================
# Cell 3 — Inspect parent/response relationships
# ============================================

print("Tweets with a parent:")
print((df["in_response_to_tweet_id"] != "").sum())

print("\nTweets with responses:")
print((df["response_tweet_id"] != "").sum())

print("\nExamples of response relationships:\n")

print(
    df.loc[
        df["response_tweet_id"] != "",
        [
            "tweet_id",
            "response_tweet_id",
            "in_response_to_tweet_id"
        ]
    ].head(20).to_string(index=False)
)

Tweets with a parent:
74090

Tweets with responses:
67571

Examples of response relationships:

tweet_id response_tweet_id in_response_to_tweet_id
       1                 2                     3.0
       3                 1                     4.0
       4                 3                     5.0
       5                 4                     6.0
       6               5,7                     8.0
       8            9,6,10                        
      12          11,13,14                    15.0
      15                12                    16.0
      16                15                    17.0
      17                16                    18.0
      18                17                        
      20                19                        
      21             22,23                    24.0
      22                25                    21.0
      25                26                    22.0
      26                27                    25.0
      24                21           

In [7]:
# ============================================
# Cell 4 — Multiple response analysis
# ============================================

response_counts = (
    df["response_tweet_id"]
    .apply(lambda x: len([r.strip() for r in x.split(",") if r.strip()]))
)

print("Response count distribution:\n")
print(response_counts.value_counts().sort_index())

print("\nTweets with multiple responses:")
print((response_counts > 1).sum())

print("\nMaximum responses from one tweet:")
print(response_counts.max())

Response count distribution:

response_tweet_id
0       32429
1       57910
2        7204
3        1378
4         414
        ...  
650         1
691         1
722         1
737         1
1090        1
Name: count, Length: 114, dtype: int64

Tweets with multiple responses:
9661

Maximum responses from one tweet:
1090


In [ ]:
'''RAW DATA
twcs.csv
   │
   ├── tweet_id
   ├── parent/response relationships
   ├── author
   ├── inbound
   └── timestamp
          │
          ▼
CONVERSATION RECONSTRUCTION
          │
          ▼
JOIN WITH CLEANED CUSTOMER DATA
          │
          ▼
conversation_id
turn_number
role
text_raw
text_clean
language
quality_flag'''

In [8]:
# ============================================
# Cell 4 — Multiple response analysis
# ============================================

response_counts = (
    df["response_tweet_id"]
    .apply(
        lambda x: len(
            [r.strip() for r in x.split(",") if r.strip()]
        )
    )
)

print("Response count distribution:")
print(response_counts.value_counts().sort_index())

print("\nTweets with multiple responses:")
print((response_counts > 1).sum())

print("\nMaximum responses from one tweet:")
print(response_counts.max())

Response count distribution:
response_tweet_id
0       32429
1       57910
2        7204
3        1378
4         414
        ...  
650         1
691         1
722         1
737         1
1090        1
Name: count, Length: 114, dtype: int64

Tweets with multiple responses:
9661

Maximum responses from one tweet:
1090


In [9]:
# ============================================
# Cell 5 — Inspect extreme response case
# ============================================

max_responses = response_counts.max()

extreme_tweet = df.loc[
    response_counts == max_responses,
    [
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "response_tweet_id"
    ]
]

print("Maximum response count:", max_responses)
print("\nTweet(s) with maximum responses:")
print(extreme_tweet.to_string(index=False))

print("\nNumber of response IDs:")
for value in extreme_tweet["response_tweet_id"]:
    ids = [x.strip() for x in value.split(",") if x.strip()]
    print(len(ids))
    print("First 20 IDs:", ids[:20])
    

Maximum response count: 1090

Tweet(s) with maximum responses:
tweet_id author_id  inbound                created_at                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [11]:
# ============================================
# Cell 7 — Fix parent tweet IDs
# ============================================

df["in_response_to_tweet_id"] = (
    df["in_response_to_tweet_id"]
    .replace("", pd.NA)
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .fillna("")
)

print(
    df[
        ["tweet_id", "in_response_to_tweet_id"]
    ].head(10).to_string(index=False)
)

tweet_id in_response_to_tweet_id
       1                       3
       2                       1
       3                       4
       4                       5
       5                       6
       6                       8
       8                        
      11                      12
      12                      15
      15                      16


In [12]:
# ============================================
# Cell 8 — Re-check parent relationships
# ============================================

tweet_ids = set(df["tweet_id"])

parent_ids = set(
    df.loc[
        df["in_response_to_tweet_id"] != "",
        "in_response_to_tweet_id"
    ]
)

existing_parents = parent_ids.intersection(tweet_ids)
missing_parents = parent_ids - tweet_ids

print("Tweets in dataset:", len(tweet_ids))
print("Unique parent IDs referenced:", len(parent_ids))
print("Parent IDs present in dataset:", len(existing_parents))
print("Parent IDs missing from dataset:", len(missing_parents))

print("\nPercentage of referenced parents available:")

if len(parent_ids) > 0:
    print(
        round(
            len(existing_parents) / len(parent_ids) * 100,
            2
        ),
        "%"
    )

Tweets in dataset: 100000
Unique parent IDs referenced: 67666
Parent IDs present in dataset: 67571
Parent IDs missing from dataset: 95

Percentage of referenced parents available:
99.86 %


In [13]:
# ============================================
# Cell 9 — Create parent lookup
# ============================================

parent_map = (
    df.set_index("tweet_id")["in_response_to_tweet_id"]
    .to_dict()
)

print("Parent map size:", len(parent_map))

print("\nSample parent relationships:")

for tweet_id in list(parent_map.keys())[:10]:
    print(
        f"Tweet {tweet_id} → "
        f"Parent {parent_map[tweet_id]}"
    )

Parent map size: 100000

Sample parent relationships:
Tweet 1 → Parent 3
Tweet 2 → Parent 1
Tweet 3 → Parent 4
Tweet 4 → Parent 5
Tweet 5 → Parent 6
Tweet 6 → Parent 8
Tweet 8 → Parent 
Tweet 11 → Parent 12
Tweet 12 → Parent 15
Tweet 15 → Parent 16


In [14]:
# ============================================
# Cell 10 — Identify AppleSupport tweets
# ============================================

apple_support = df[
    df["author_id"].str.lower() == "applesupport"
].copy()

print("AppleSupport tweets:", len(apple_support))

print("\nFirst 10 AppleSupport tweets:")

print(
    apple_support[
        [
            "tweet_id",
            "created_at",
            "in_response_to_tweet_id",
            "response_tweet_id",
            "text"
        ]
    ].head(10).to_string(index=False)
)

AppleSupport tweets: 3106

First 10 AppleSupport tweets:
tweet_id                created_at in_response_to_tweet_id response_tweet_id                                                                                                                                           text
     696 2017-10-31 22:27:49+00:00                     698               697                             @115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.
     699 2017-10-31 22:36:27+00:00                     697                   @115854 Lets take a closer look into this issue. Select the following link to join us in a DM and we'll go from there. https://t.co/GDrqU22YpT
     701 2017-10-31 22:26:49+00:00                     702                                                                                   @115855 Let's go to DM for the next steps. DM us here: https://t.co/GDrqU22YpT
     703 2017-10-31 22:09:52+00:00                     704     

In [15]:
# ============================================
# Cell 11 — Trace one conversation upward
# ============================================

def trace_parents(tweet_id, parent_map, max_depth=20):
    chain = []
    current = str(tweet_id)

    for _ in range(max_depth):
        if not current:
            break

        chain.append(current)

        parent = parent_map.get(current, "")

        if not parent:
            break

        if parent in chain:
            print("Cycle detected.")
            break

        current = parent

    return chain


tweet_id = "696"

chain = trace_parents(
    tweet_id,
    parent_map
)

print("Parent chain for tweet", tweet_id)
print(" → ".join(chain))

print("\nTweet details:\n")

for tid in chain:
    row = df[df["tweet_id"] == tid]

    if len(row) == 0:
        print(f"{tid}: NOT FOUND")
    else:
        row = row.iloc[0]

        print(
            f"Tweet {tid} | "
            f"Author: {row['author_id']} | "
            f"Inbound: {row['inbound']}"
        )
        print(row["text"])
        print("-" * 80)
        

Parent chain for tweet 696
696 → 698 → 700

Tweet details:

Tweet 696 | Author: AppleSupport | Inbound: False
@115854 We're here for you. Which version of the iOS are you running? Check from Settings &gt; General &gt; About.
--------------------------------------------------------------------------------
Tweet 698 | Author: 115854 | Inbound: True
@AppleSupport  https://t.co/NV0yucs0lB
--------------------------------------------------------------------------------
Tweet 700 | Author: 115854 | Inbound: True
@AppleSupport why are my I️’s changing not showing up correctly on any of my social media platforms? https://t.co/GyRvpyVnkE
--------------------------------------------------------------------------------


In [16]:
# ============================================
# Cell 12 — Inspect a multi-response conversation
# ============================================

tweet_id = "705"

row = df[df["tweet_id"] == tweet_id].iloc[0]

print("Parent tweet:")
print(
    f"Tweet {row['tweet_id']} | "
    f"Author: {row['author_id']} | "
    f"Inbound: {row['inbound']}"
)
print(row["text"])

print("\nResponse IDs:")
print(row["response_tweet_id"])

response_ids = [
    x.strip()
    for x in row["response_tweet_id"].split(",")
    if x.strip()
]

print("\nChild tweets:\n")

for response_id in response_ids:

    child = df[df["tweet_id"] == response_id]

    if len(child) == 0:
        print(f"Tweet {response_id}: NOT FOUND")
        continue

    child = child.iloc[0]

    print(
        f"Tweet {child['tweet_id']} | "
        f"Author: {child['author_id']} | "
        f"Inbound: {child['inbound']} | "
        f"Parent: {child['in_response_to_tweet_id']}"
    )

    print(child["text"])
    print("-" * 80)

Parent tweet:
Tweet 705 | Author: AppleSupport | Inbound: False
@115855 That's great it has iOS 11.1 as we can rule out being outdated. Any steps tried since this started? Do you recall when it started?

Response IDs:
706,704

Child tweets:

Tweet 706: NOT FOUND
Tweet 704 | Author: 115855 | Inbound: True | Parent: 705
@AppleSupport This is what it looks like https://t.co/XCQU2l4xUB
--------------------------------------------------------------------------------


In [17]:
# ============================================
# Cell 13 — Validate response IDs against dataset
# ============================================

tweet_ids = set(df["tweet_id"])

total_response_ids = 0
existing_response_ids = 0
missing_response_ids = 0

for value in df["response_tweet_id"]:
    ids = [
        x.strip()
        for x in value.split(",")
        if x.strip()
    ]

    total_response_ids += len(ids)

    for response_id in ids:
        if response_id in tweet_ids:
            existing_response_ids += 1
        else:
            missing_response_ids += 1

print("Total response IDs:", total_response_ids)
print("Response tweets present:", existing_response_ids)
print("Response tweets missing:", missing_response_ids)

print(
    "\nResponse IDs available in current sample:",
    round(
        existing_response_ids / total_response_ids * 100,
        2
    ),
    "%"
)

Total response IDs: 100025
Response tweets present: 73995
Response tweets missing: 26030

Response IDs available in current sample: 73.98 %


In [18]:
# ============================================
# Cell 14 — Build child lookup
# ============================================

child_map = {}

for _, row in df.iterrows():
    parent_id = row["in_response_to_tweet_id"]

    if parent_id:
        child_map.setdefault(parent_id, []).append(
            row["tweet_id"]
        )

print("Tweets with at least one child in sample:", len(child_map))

print("\nExample child relationships:")

for parent_id, children in list(child_map.items())[:10]:
    print(
        f"Parent {parent_id} → Children {children}"
    )

Tweets with at least one child in sample: 67666

Example child relationships:
Parent 3 → Children ['1']
Parent 1 → Children ['2']
Parent 4 → Children ['3']
Parent 5 → Children ['4']
Parent 6 → Children ['5']
Parent 8 → Children ['6']
Parent 12 → Children ['11', '13']
Parent 15 → Children ['12']
Parent 16 → Children ['15']
Parent 17 → Children ['16']


In [19]:
# ============================================
# Cell 15 — Assign conversation root IDs
# ============================================

from functools import lru_cache

@lru_cache(maxsize=None)
def find_root(tweet_id):
    current = str(tweet_id)
    visited = set()

    while current:
        if current in visited:
            # Safety against unexpected cycles
            return current

        visited.add(current)

        parent = parent_map.get(current, "")

        if not parent:
            return current

        # Parent exists in our sample
        if parent not in parent_map:
            return current

        current = parent

    return str(tweet_id)


df["conversation_id"] = df["tweet_id"].apply(find_root)

print("Unique conversations:", df["conversation_id"].nunique())

print("\nExample conversation assignments:")
print(
    df[
        ["tweet_id", "in_response_to_tweet_id", "conversation_id"]
    ]
    .head(20)
    .to_string(index=False)
)

Unique conversations: 26005

Example conversation assignments:
tweet_id in_response_to_tweet_id conversation_id
       1                       3               8
       2                       1               8
       3                       4               8
       4                       5               8
       5                       6               8
       6                       8               8
       8                                       8
      11                      12              18
      12                      15              18
      15                      16              18
      16                      17              18
      17                      18              18
      18                                      18
      19                      20              20
      20                                      20
      21                      24              29
      22                      21              29
      25                      22              29
      

In [20]:
# ============================================
# Cell 16 — Inspect one reconstructed conversation
# ============================================

conversation_id = "8"

conversation = (
    df[df["conversation_id"] == conversation_id]
    .sort_values("created_at")
    .copy()
)

print("Conversation ID:", conversation_id)
print("Number of tweets:", len(conversation))

print("\n" + "=" * 100)

for _, row in conversation.iterrows():
    role = (
        "CUSTOMER"
        if row["inbound"]
        else "SUPPORT"
    )

    print(
        f"[{row['created_at']}] "
        f"{role} | Tweet {row['tweet_id']}"
    )
    print(row["text"])
    print("-" * 100)

Conversation ID: 8
Number of tweets: 7

[2017-10-31 21:45:10+00:00] CUSTOMER | Tweet 8
@sprintcare is the worst customer service
----------------------------------------------------------------------------------------------------
[2017-10-31 21:46:24+00:00] SUPPORT | Tweet 6
@115712 Can you please send us a private message, so that I can gain further details about your account?
----------------------------------------------------------------------------------------------------
[2017-10-31 21:49:35+00:00] CUSTOMER | Tweet 5
@sprintcare I did.
----------------------------------------------------------------------------------------------------
[2017-10-31 21:54:49+00:00] SUPPORT | Tweet 4
@115712 Please send us a Private Message so that we can further assist you. Just click ‘Message’ at the top of your profile.
----------------------------------------------------------------------------------------------------
[2017-10-31 22:08:27+00:00] CUSTOMER | Tweet 3
@sprintcare I have sent several 

In [21]:
# ============================================
# Cell 17 — Identify AppleSupport conversations
# ============================================

apple_support_conversation_ids = set(
    df.loc[
        df["author_id"].str.lower() == "applesupport",
        "conversation_id"
    ]
)

apple_conversations = df[
    df["conversation_id"].isin(apple_support_conversation_ids)
].copy()

print("AppleSupport conversations:", len(apple_support_conversation_ids))
print("Tweets belonging to AppleSupport conversations:", len(apple_conversations))

print("\nExample conversation IDs:")
print(list(apple_support_conversation_ids)[:20])

print("\nRole distribution:")
print(
    apple_conversations["inbound"]
    .map({True: "CUSTOMER", False: "SUPPORT"})
    .value_counts()
)

AppleSupport conversations: 2066
Tweets belonging to AppleSupport conversations: 6836

Example conversation IDs:
['4906', '31491', '79997', '62254', '13953', '51623', '12010', '44571', '41558', '65010', '107371', '105349', '114521', '31897', '124440', '125030', '59606', '92724', '92661', '61342']

Role distribution:
inbound
CUSTOMER    3729
SUPPORT     3107
Name: count, dtype: int64


In [22]:
# ============================================
# Cell 18 — Inspect one AppleSupport conversation
# ============================================

conversation_id = next(iter(apple_support_conversation_ids))

conversation = (
    apple_conversations[
        apple_conversations["conversation_id"] == conversation_id
    ]
    .sort_values("created_at")
    .copy()
)

print("Conversation ID:", conversation_id)
print("Number of tweets:", len(conversation))

print("\n" + "=" * 100)

for _, row in conversation.iterrows():

    role = (
        "CUSTOMER"
        if row["inbound"]
        else "SUPPORT"
    )

    print(
        f"[{row['created_at']}] "
        f"{role} | Tweet {row['tweet_id']}"
    )

    print(row["text"])

    print("-" * 100)

Conversation ID: 4906
Number of tweets: 4

[2017-10-31 23:06:38+00:00] CUSTOMER | Tweet 4906
@AppleSupport The ‘Overlapping url over TIME on Free WiFi hotspots’ remains. v11.0.3. Not a WiFi problem; two different hotspots shown. https://t.co/81ifcrDUeS
----------------------------------------------------------------------------------------------------
[2017-10-31 23:18:26+00:00] SUPPORT | Tweet 4904
@116855 We'd like to have you backup and update your iPhone to iOS 11.1 and let us know if that helps: https://t.co/ahjigcvFRG
----------------------------------------------------------------------------------------------------
[2017-10-31 23:21:45+00:00] CUSTOMER | Tweet 4905
@AppleSupport I've updated to iOS 11.1 just now. I shall check out these WiFi hotspots tomorrow and later in the week.
----------------------------------------------------------------------------------------------------
[2017-10-31 23:34:00+00:00] SUPPORT | Tweet 4907
@116855 Sounds good. Let us know the results in DM

In [23]:
# ============================================
# Cell 19 — Build structured conversation data
# ============================================

conversation_df = (
    apple_conversations
    .sort_values(["conversation_id", "created_at"])
    .copy()
)

conversation_df["role"] = conversation_df["inbound"].map({
    True: "customer",
    False: "support"
})

conversation_df["turn_number"] = (
    conversation_df
    .groupby("conversation_id")
    .cumcount() + 1
)

conversation_df = conversation_df[
    [
        "conversation_id",
        "turn_number",
        "tweet_id",
        "author_id",
        "role",
        "created_at",
        "text",
        "in_response_to_tweet_id",
        "response_tweet_id"
    ]
].rename(
    columns={
        "text": "text_raw",
        "in_response_to_tweet_id": "parent_tweet_id",
        "response_tweet_id": "response_tweet_ids"
    }
)

print("Shape:", conversation_df.shape)

print("\nColumns:")
print(conversation_df.columns.tolist())

print("\nRole distribution:")
print(conversation_df["role"].value_counts())

print("\nExample:")
print(
    conversation_df[
        conversation_df["conversation_id"] == "4906"
    ].to_string(index=False)
)

Shape: (6836, 9)

Columns:
['conversation_id', 'turn_number', 'tweet_id', 'author_id', 'role', 'created_at', 'text_raw', 'parent_tweet_id', 'response_tweet_ids']

Role distribution:
role
customer    3729
support     3107
Name: count, dtype: int64

Example:
conversation_id  turn_number tweet_id    author_id     role                created_at                                                                                                                                                        text_raw parent_tweet_id response_tweet_ids
           4906            1     4906       116855 customer 2017-10-31 23:06:38+00:00 @AppleSupport The ‘Overlapping url over TIME on Free WiFi hotspots’ remains. v11.0.3. Not a WiFi problem; two different hotspots shown. https://t.co/81ifcrDUeS                               4904
           4906            2     4904 AppleSupport  support 2017-10-31 23:18:26+00:00                                  @116855 We'd like to have you backup and update your iPhone to

In [24]:
# ============================================
# Cell 20 — Conversation quality statistics
# ============================================

conversation_stats = (
    conversation_df
    .groupby("conversation_id")
    .agg(
        turns=("tweet_id", "count"),
        customer_turns=("role", lambda x: (x == "customer").sum()),
        support_turns=("role", lambda x: (x == "support").sum())
    )
)

print("Total conversations:", len(conversation_stats))

print("\nConversation length distribution:")
print(
    conversation_stats["turns"]
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nConversations with multiple turns:")
print(
    (conversation_stats["turns"] > 1).sum()
)

print("\nConversations with customer + support:")
print(
    (
        (conversation_stats["customer_turns"] > 0) &
        (conversation_stats["support_turns"] > 0)
    ).sum()
)

print("\nAverage turns per conversation:")
print(round(conversation_stats["turns"].mean(), 2))

print("\nMaximum turns in one conversation:")
print(conversation_stats["turns"].max())

Total conversations: 2066

Conversation length distribution:
turns
2     1149
3      177
4      362
5      111
6      126
7       41
8       52
9       22
10      13
11       2
12       5
13       1
17       2
24       1
27       1
37       1
Name: count, dtype: int64

Conversations with multiple turns:
2066

Conversations with customer + support:
2066

Average turns per conversation:
3.31

Maximum turns in one conversation:
37


In [25]:
# ============================================
# Cell 21 — Inspect long conversations
# ============================================

long_conversations = (
    conversation_stats[
        conversation_stats["turns"] >= 10
    ]
    .sort_values("turns", ascending=False)
)

print("Conversations with 10+ turns:", len(long_conversations))

print("\nLongest conversations:")
print(long_conversations)

print("\n" + "=" * 100)

for conversation_id in long_conversations.head(3).index:

    conversation = (
        conversation_df[
            conversation_df["conversation_id"] == conversation_id
        ]
        .sort_values("turn_number")
    )

    print(
        f"\nConversation {conversation_id} "
        f"({len(conversation)} turns)"
    )

    for _, row in conversation.iterrows():
        print(
            f"{row['turn_number']}. "
            f"[{row['role'].upper()}] "
            f"{row['text_raw'][:250]}"
        )

    print("-" * 100)

Conversations with 10+ turns: 26

Longest conversations:
                 turns  customer_turns  support_turns
conversation_id                                      
9200                37              20             17
117940              27              18              9
40514               24              21              3
33360               17               9              8
61348               17               9              8
61302               13               7              6
50818               12               7              5
39604               12               8              4
120658              12               9              3
62279               12               7              5
87091               12               6              6
38645               11               6              5
40527               11               6              5
110869              10               5              5
34598               10               5              5
125764              10   

In [26]:
# ============================================
# Cell 22 — Inspect longest conversation
# ============================================

conversation_id = "9200"

conversation = (
    conversation_df[
        conversation_df["conversation_id"] == conversation_id
    ]
    .sort_values("turn_number")
)

print("Conversation ID:", conversation_id)
print("Total turns:", len(conversation))

print("\n" + "=" * 100)

for _, row in conversation.iterrows():

    print(
        f"Turn {row['turn_number']} | "
        f"{row['role'].upper()} | "
        f"Tweet {row['tweet_id']} | "
        f"Parent {row['parent_tweet_id']}"
    )

    print(row["text_raw"])

    print("-" * 100)

Conversation ID: 9200
Total turns: 37

Turn 1 | SUPPORT | Tweet 9200 | Parent 
It’s attack of the new emoji. 

Unleash a zombie emoji-pocalypse this Halloween with the echo effect in Messages. https://t.co/P4l8dMu9dj
----------------------------------------------------------------------------------------------------
Turn 2 | CUSTOMER | Tweet 9198 | Parent 9200
@AppleSupport Does the person you send it too also have to have a iPhone to see this?I see it on my phone.
----------------------------------------------------------------------------------------------------
Turn 3 | SUPPORT | Tweet 9196 | Parent 9198
@117654 Great question. They must have a device with iOS 10 or later. Here's some more information: https://t.co/GCtGkHihKu
----------------------------------------------------------------------------------------------------
Turn 4 | CUSTOMER | Tweet 9197 | Parent 9196
@AppleSupport But would it work with another phone brand?Like LG galaxy phones?
-----------------------------------

In [27]:
# ============================================
# Cell 23 — Measure conversation branching
# ============================================

branching = (
    conversation_df
    .groupby("parent_tweet_id")
    .size()
    .sort_values(ascending=False)
)

branching = branching[branching.index != ""]

print("Tweets with multiple direct replies:")
print((branching > 1).sum())

print("\nMaximum direct replies to one tweet:")
print(branching.max())

print("\nTop 20 most-replied-to tweets:")

print(
    branching.head(20).to_string()
)

Tweets with multiple direct replies:
150

Maximum direct replies to one tweet:
11

Top 20 most-replied-to tweets:
parent_tweet_id
9200      11
117940     7
33360      6
39661      6
40491      6
125796     4
117939     4
53607      4
61348      4
12573      3
39605      3
29554      3
50051      3
55938      3
90550      3
94772      3
23095      2
37656      2
39649      2
38869      2


In [30]:
# ============================================
# Cell 25 — Correct AppleSupport branching analysis
# ============================================

apple_parent_counts = (
    conversation_df[
        conversation_df["parent_tweet_id"] != ""
    ]
    .groupby("parent_tweet_id")
    .size()
    .sort_values(ascending=False)
)

apple_branch_points = apple_parent_counts[
    apple_parent_counts > 1
]

print(
    "Parent tweets with multiple direct replies:",
    len(apple_branch_points)
)

print(
    "\nMaximum direct replies:",
    apple_parent_counts.max()
    if len(apple_parent_counts) > 0
    else 0
)

print("\nTop branching parent tweets:")
print(
    apple_branch_points.head(20).to_string()
)

Parent tweets with multiple direct replies: 150

Maximum direct replies: 11

Top branching parent tweets:
parent_tweet_id
9200      11
117940     7
33360      6
39661      6
40491      6
117939     4
125796     4
53607      4
61348      4
39605      3
29554      3
12573      3
50051      3
55938      3
90550      3
94772      3
125806     2
23095      2
37656      2
39649      2


In [31]:
# ============================================
# Cell 26 — Mark branching tweets
# ============================================

conversation_df["direct_reply_count"] = (
    conversation_df["tweet_id"]
    .map(apple_parent_counts)
    .fillna(0)
    .astype(int)
)

conversation_df["is_branch_point"] = (
    conversation_df["direct_reply_count"] > 1
)

print(
    "Tweets that are branch points:",
    conversation_df["is_branch_point"].sum()
)

print("\nBranching tweets:")
print(
    conversation_df[
        conversation_df["is_branch_point"]
    ][
        [
            "conversation_id",
            "tweet_id",
            "role",
            "direct_reply_count",
            "text_raw"
        ]
    ]
    .sort_values("direct_reply_count", ascending=False)
    .head(20)
    .to_string(index=False)
)

Tweets that are branch points: 150

Branching tweets:
conversation_id tweet_id     role  direct_reply_count                                                                                                                                                                                                                                                      text_raw
           9200     9200  support                  11                                                                                                                   It’s attack of the new emoji. \n\nUnleash a zombie emoji-pocalypse this Halloween with the echo effect in Messages. https://t.co/P4l8dMu9dj
         117940   117940 customer                   7                                                                                              Hi @AppleSupport, could you pls explain why, since I downloaded yr iOS update, my iPhone crashes at 10%,  20%, even 40% battery? Massively inconvenient at best.
          40514    404

In [32]:
# ============================================
# Cell 27 — Inspect processed data files
# ============================================

from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

print("Processed directory:", PROCESSED_DIR.resolve())

if not PROCESSED_DIR.exists():
    print("\nERROR: data/processed/ does not exist.")
else:
    files = sorted(
        [
            p for p in PROCESSED_DIR.rglob("*")
            if p.is_file()
        ]
    )

    print("\nFiles found:", len(files))

    for file in files:
        print(file.relative_to(PROCESSED_DIR))

Processed directory: D:\AgentCustomerSupport\hiver-sde-intern\data\processed

Files found: 1
.gitkeep


In [35]:
# ============================================
# Cell 29 — Load Notebook 01 cleaned dataset
# ============================================

cleaned_customers = pd.read_csv(
    "../data/processed/apple_customer_tweets_cleaned.csv"
)

print("Cleaned customer dataset shape:", cleaned_customers.shape)

print("\nColumns:")
print(cleaned_customers.columns.tolist())

print("\nSample:")
print(
    cleaned_customers[
        ["tweet_id", "text_clean", "language", "quality_flag"]
    ].head().to_string(index=False)
)

Cleaned customer dataset shape: (3020, 11)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'text_raw', 'text_clean', 'language', 'quality_flag']

Sample:
 tweet_id                                                                            text_clean language         quality_flag
      697                              The newest update. I made sure to download it yesterday.       en                 KEEP
      698                                                                                   NaN  unknown EMPTY_AFTER_CLEANING
      700 why are my I’s changing not showing up correctly on any of my social media platforms?       en                 KEEP
      702                        Tried resetting my settings .. restarting my phone .. all that       en                 KEEP
      704                                                            This is what it looks like       en                 KEEP


In [34]:
# ============================================
# Cell 29 — Validate cleaned text
# ============================================

print("Total tweets:", len(conversation_df))

print(
    "\nEmpty cleaned texts:",
    (conversation_df["text_clean"] == "").sum()
)

print(
    "Texts changed by cleaning:",
    (
        conversation_df["text_raw"]
        != conversation_df["text_clean"]
    ).sum()
)

print("\nCleaning examples:")

changed = conversation_df[
    conversation_df["text_raw"]
    != conversation_df["text_clean"]
]

print(
    changed[
        ["tweet_id", "role", "text_raw", "text_clean"]
    ]
    .head(10)
    .to_string(index=False)
)

Total tweets: 6836

Empty cleaned texts: 36
Texts changed by cleaning: 6820

Cleaning examples:
tweet_id     role                                                                                                                                                                                                                                                                                                           text_raw                                                                                                                                                                                                                                                              text_clean
  100245 customer                                                    What is this @AppleSupport? since I upgraded with this $hi&amp;*y #macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Mac

In [36]:
# ============================================
# Cell 30 — Load cleaned customer dataset
# ============================================

cleaned_customers = pd.read_csv(
    "../data/processed/apple_customer_tweets_cleaned.csv"
)

print("Cleaned customer dataset shape:", cleaned_customers.shape)

print("\nColumns:")
print(cleaned_customers.columns.tolist())

print("\nSample:")
print(
    cleaned_customers[
        ["tweet_id", "text_clean", "language", "quality_flag"]
    ].head().to_string(index=False)
)

Cleaned customer dataset shape: (3020, 11)

Columns:
['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id', 'text_raw', 'text_clean', 'language', 'quality_flag']

Sample:
 tweet_id                                                                            text_clean language         quality_flag
      697                              The newest update. I made sure to download it yesterday.       en                 KEEP
      698                                                                                   NaN  unknown EMPTY_AFTER_CLEANING
      700 why are my I’s changing not showing up correctly on any of my social media platforms?       en                 KEEP
      702                        Tried resetting my settings .. restarting my phone .. all that       en                 KEEP
      704                                                            This is what it looks like       en                 KEEP


In [38]:
# ============================================
# Cell 31 — Attach Notebook 01 cleaning fields
# ============================================

# Normalize tweet IDs to strings on both datasets
conversation_df["tweet_id"] = (
    conversation_df["tweet_id"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

cleaned_customers["tweet_id"] = (
    cleaned_customers["tweet_id"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

# Keep only the cleaning fields from Notebook 01
cleaning_fields = cleaned_customers[
    ["tweet_id", "text_clean", "language", "quality_flag"]
].copy()

# Remove the temporary text_clean created earlier in Notebook 02
conversation_df = conversation_df.drop(
    columns=["text_clean"],
    errors="ignore"
)

# Merge Notebook 01 cleaning results
conversation_df = conversation_df.merge(
    cleaning_fields,
    on="tweet_id",
    how="left"
)

print("Conversation shape:", conversation_df.shape)

print("\nCleaning fields:")
print(
    conversation_df[
        ["tweet_id", "role", "text_raw", "text_clean",
         "language", "quality_flag"]
    ].head(10).to_string(index=False)
)

print("\nMissing cleaning fields:")
print(
    conversation_df[
        ["text_clean", "language", "quality_flag"]
    ].isna().sum()
)

Conversation shape: (6836, 14)

Cleaning fields:
tweet_id     role                                                                                                                                                                                                                                                                                                           text_raw                                                                                                                                                                                                                                    text_clean language quality_flag
  100245 customer                                                    What is this @AppleSupport? since I upgraded with this $hi&amp;*y #macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW! What is this ? since I upgraded with this

In [39]:
# ============================================
# Cell 32 — Check customer coverage
# ============================================

customer_rows = conversation_df[
    conversation_df["role"] == "customer"
].copy()

missing_customer_cleaning = customer_rows[
    customer_rows["quality_flag"].isna()
]

print("Total customer tweets in conversations:",
      len(customer_rows))

print("Customer tweets with Notebook 01 cleaning:",
      customer_rows["quality_flag"].notna().sum())

print("Customer tweets missing Notebook 01 cleaning:",
      len(missing_customer_cleaning))

print("\nMissing customer examples:")

print(
    missing_customer_cleaning[
        ["tweet_id", "text_raw"]
    ].head(20).to_string(index=False)
)

Total customer tweets in conversations: 3729
Customer tweets with Notebook 01 cleaning: 2998
Customer tweets missing Notebook 01 cleaning: 731

Missing customer examples:
tweet_id                                                                                                                                                                                                                                                                                text_raw
  101231                                                                                                                                                                                              Noticed a bug on my @115858 iPhone X/iOS 11.1.2 While I’m on the phone I can’t close apps.
  101239                                                                                            @115858 Just because you're releasing new phones doesn't mean you should stop making my current iPhone stop working. I have a contract and cannot just 

In [40]:
# ============================================
# Cell 33 — Validate Notebook 01 cleaning coverage
# ============================================

customer_rows = conversation_df[
    conversation_df["role"] == "customer"
].copy()

covered_customers = customer_rows[
    customer_rows["quality_flag"].notna()
]

uncovered_customers = customer_rows[
    customer_rows["quality_flag"].isna()
]

print("Customer tweets in reconstructed conversations:",
      len(customer_rows))

print("Customer tweets covered by Notebook 01:",
      len(covered_customers))

print("Customer tweets not covered by Notebook 01:",
      len(uncovered_customers))

print(
    "\nCoverage:",
    round(len(covered_customers) / len(customer_rows) * 100, 2),
    "%"
)

print("\nUncovered tweets are retained as raw conversation context.")

Customer tweets in reconstructed conversations: 3729
Customer tweets covered by Notebook 01: 2998
Customer tweets not covered by Notebook 01: 731

Coverage: 80.4 %

Uncovered tweets are retained as raw conversation context.


In [41]:
# ============================================
# Cell 34 — Branch-aware conversation check
# ============================================

branch_summary = (
    conversation_df[
        conversation_df["direct_reply_count"] > 1
    ]
    .groupby("conversation_id")
    .agg(
        branch_points=("tweet_id", "count"),
        max_direct_replies=("direct_reply_count", "max"),
        total_tweets=("tweet_id", "size")
    )
    .sort_values(
        ["max_direct_replies", "total_tweets"],
        ascending=False
    )
)

print("Conversation components containing branch points:",
      len(branch_summary))

print("\nTop branching components:")

print(
    branch_summary.head(15).to_string()
)

Conversation components containing branch points: 132

Top branching components:
                 branch_points  max_direct_replies  total_tweets
conversation_id                                                 
9200                         1                  11             1
117940                       3                   7             3
40514                        3                   6             3
33360                        1                   6             1
39668                        1                   6             1
125801                       1                   4             1
53612                        1                   4             1
61348                        1                   4             1
29558                        2                   3             2
39604                        2                   3             2
12577                        1                   3             1
50050                        1                   3             1
55942    

In [42]:
# ============================================
# Cell 35 — Inspect a branching component
# ============================================

branch_id = "9200"

branch_example = conversation_df[
    conversation_df["conversation_id"] == branch_id
].copy()

print("Tweets currently inside component:", len(branch_example))

print("\nTweets:")
print(
    branch_example[
        [
            "tweet_id",
            "role",
            "text_raw",
            "parent_tweet_id",
            "direct_reply_count",
            "is_branch_point"
        ]
    ].to_string(index=False)
)

print("\nAvailable direct children from child_map:")

children = child_map.get(branch_id, [])

print("Number of children:", len(children))
print("Child tweet IDs:", children)

Tweets currently inside component: 37

Tweets:
tweet_id     role                                                                                                                                                   text_raw parent_tweet_id  direct_reply_count  is_branch_point
    9200  support                It’s attack of the new emoji. \n\nUnleash a zombie emoji-pocalypse this Halloween with the echo effect in Messages. https://t.co/P4l8dMu9dj                                  11             True
    9198 customer                                                 @AppleSupport Does the person you send it too also have to have a iPhone to see this?I see it on my phone.            9200                   1            False
    9196  support                                @117654 Great question. They must have a device with iOS 10 or later. Here's some more information: https://t.co/GCtGkHihKu            9198                   1            False
    9197 customer                                

In [43]:
# ============================================
# Cell 36 — Validate graph coverage for AppleSupport
# ============================================

conversation_tweet_ids = set(
    conversation_df["tweet_id"].astype(str)
)

graph_children_in_subset = 0
graph_children_outside_subset = 0

for parent_id, children in child_map.items():
    if parent_id in conversation_tweet_ids:
        for child_id in children:
            if child_id in conversation_tweet_ids:
                graph_children_in_subset += 1
            else:
                graph_children_outside_subset += 1

print("Parent-child edges inside AppleSupport subset:",
      graph_children_in_subset)

print("Parent-child edges pointing outside subset:",
      graph_children_outside_subset)

print(
    "\nTotal parent-child edges from AppleSupport parents:",
    graph_children_in_subset + graph_children_outside_subset
)

Parent-child edges inside AppleSupport subset: 4770
Parent-child edges pointing outside subset: 0

Total parent-child edges from AppleSupport parents: 4770


In [44]:
# ============================================
# Cell 37 — Build branch-aware thread IDs
# ============================================

from collections import defaultdict

# Map each tweet to its children that are actually
# present in the AppleSupport conversation subset
subset_children = defaultdict(list)

for _, row in conversation_df.iterrows():
    parent_id = row["parent_tweet_id"]

    if parent_id and parent_id in conversation_tweet_ids:
        subset_children[parent_id].append(row["tweet_id"])


def build_threads(df):
    """
    Create branch-aware thread IDs from the actual
    parent-child graph.

    A new thread starts from a root tweet or from
    every child of a branching parent.
    """

    thread_records = []

    for root_id, group in df.groupby("conversation_id"):
        group_ids = set(group["tweet_id"])

        # Find the root tweet(s) of this component
        roots = [
            tweet_id
            for tweet_id in group_ids
            if not group.loc[
                group["tweet_id"] == tweet_id,
                "parent_tweet_id"
            ].iloc[0]
        ]

        # DFS through the graph
        stack = []

        for root in roots:
            stack.append((root, [root]))

        while stack:
            current, path = stack.pop()

            children = [
                child
                for child in subset_children.get(current, [])
                if child in group_ids
            ]

            if not children:
                thread_records.append({
                    "conversation_id": root_id,
                    "thread_path": path
                })
            else:
                for child in children:
                    stack.append(
                        (child, path + [child])
                    )

    return thread_records


thread_records = build_threads(conversation_df)

print("Branch-aware threads created:", len(thread_records))

print("\nSample thread paths:")

for record in thread_records[:10]:
    print(
        record["conversation_id"],
        "→",
        " → ".join(record["thread_path"])
    )

Branch-aware threads created: 2247

Sample thread paths:
100245 → 100245 → 100244 → 100243 → 100241 → 100242
100540 → 100540 → 100539
101228 → 101228 → 101227
101231 → 101231 → 101230
101234 → 101234 → 101232 → 101233 → 101235 → 101236 → 101237
101239 → 101239 → 101238
101241 → 101241 → 101240
101244 → 101244 → 101243
101246 → 101246 → 101245
101249 → 101249 → 101248 → 101247


In [45]:
# ============================================
# Cell 38 — Validate branch-aware threads
# ============================================

threads_9200 = [
    record
    for record in thread_records
    if record["conversation_id"] == "9200"
]

print("Threads generated from component 9200:",
      len(threads_9200))

print("\nThread paths from 9200:")

for i, record in enumerate(threads_9200, start=1):
    print(
        f"Thread {i}:",
        " → ".join(record["thread_path"])
    )

Threads generated from component 9200: 11

Thread paths from 9200:
Thread 1: 9200 → 9239 → 59602 → 59603 → 59604
Thread 2: 9200 → 9233 → 44617 → 44618 → 44619
Thread 3: 9200 → 9229 → 36075
Thread 4: 9200 → 9223 → 38209 → 38208
Thread 5: 9200 → 9221 → 34691
Thread 6: 9200 → 9275 → 33327 → 33328 → 33329
Thread 7: 9200 → 9217 → 34658 → 34659 → 34660
Thread 8: 9200 → 9214 → 32370 → 32371 → 32372
Thread 9: 9200 → 9212 → 32356 → 32357
Thread 10: 9200 → 9272 → 13192
Thread 11: 9200 → 9198 → 9196 → 9197 → 9199


In [46]:
# ============================================
# Cell 39 — Attach branch-aware thread IDs
# ============================================

thread_map = {}

for index, record in enumerate(thread_records, start=1):

    thread_id = (
        f"{record['conversation_id']}_thread_{index}"
    )

    for position, tweet_id in enumerate(record["thread_path"], start=1):
        thread_map[tweet_id] = {
            "thread_id": thread_id,
            "thread_position": position
        }

# Attach thread metadata
conversation_df["thread_id"] = (
    conversation_df["tweet_id"]
    .map(lambda x: thread_map.get(x, {}).get("thread_id"))
)

conversation_df["thread_position"] = (
    conversation_df["tweet_id"]
    .map(lambda x: thread_map.get(x, {}).get("thread_position"))
)

print("Unique branch-aware threads:",
      conversation_df["thread_id"].nunique())

print(
    "\nMissing thread IDs:",
    conversation_df["thread_id"].isna().sum()
)

print("\nSample:")
print(
    conversation_df[
        [
            "conversation_id",
            "thread_id",
            "thread_position",
            "tweet_id",
            "role",
            "text_raw"
        ]
    ].head(15).to_string(index=False)
)

Unique branch-aware threads: 2247

Missing thread IDs: 26

Sample:
conversation_id       thread_id  thread_position tweet_id     role                                                                                                                                                                                                                                                                                                             text_raw
         100245 100245_thread_1              1.0   100245 customer                                                      What is this @AppleSupport? since I upgraded with this $hi&amp;*y #macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!
         100245 100245_thread_1              2.0   100244  support                                                                                                          @137967 W

In [47]:
# ============================================
# Cell 40 — Diagnose missing thread IDs
# ============================================

missing_threads = conversation_df[
    conversation_df["thread_id"].isna()
].copy()

print("Tweets missing thread_id:",
      len(missing_threads))

print("\nMissing tweets:")

print(
    missing_threads[
        [
            "conversation_id",
            "tweet_id",
            "role",
            "parent_tweet_id",
            "text_raw"
        ]
    ].to_string(index=False)
)

print("\nAffected conversation components:")

print(
    missing_threads["conversation_id"]
    .value_counts()
    .to_string()
)

Tweets missing thread_id: 26

Missing tweets:
conversation_id tweet_id     role parent_tweet_id                                                                                                                                                                                                                                                                                   text_raw
          11651    11651  support           11653                                                                                                                                                                  @118247 Thanks for reaching out to us. Can you please let us know if you have updated to iOS 11.1? It was released today.
          11651    11650 customer           11651                                                                                                                                                                                                   @AppleSupport Just updated it and still is t

In [48]:
# ============================================
# Cell 41 — Correct branch-aware thread construction
# ============================================

from collections import defaultdict

# Build children using only tweets inside conversation_df
subset_children = defaultdict(list)

for _, row in conversation_df.iterrows():
    parent_id = row["parent_tweet_id"]
    tweet_id = row["tweet_id"]

    if (
        parent_id
        and parent_id in conversation_tweet_ids
    ):
        subset_children[parent_id].append(tweet_id)


def build_threads(df):
    """
    Build branch-aware threads.

    A thread starts when a tweet's parent is NOT present
    inside the current AppleSupport component.

    This correctly handles components whose actual parent
    exists outside the selected subset.
    """

    thread_records = []

    for component_id, group in df.groupby("conversation_id"):

        group_ids = set(group["tweet_id"])

        # A local root is a tweet whose parent is not
        # present inside this component.
        roots = []

        for _, row in group.iterrows():
            parent_id = row["parent_tweet_id"]

            if not parent_id or parent_id not in group_ids:
                roots.append(row["tweet_id"])

        # Traverse every local root
        stack = [
            (root, [root])
            for root in roots
        ]

        while stack:

            current, path = stack.pop()

            children = [
                child
                for child in subset_children.get(current, [])
                if child in group_ids
            ]

            if not children:
                thread_records.append({
                    "conversation_id": component_id,
                    "thread_path": path
                })

            else:
                for child in children:
                    stack.append(
                        (child, path + [child])
                    )

    return thread_records


thread_records = build_threads(conversation_df)

print(
    "Branch-aware threads created:",
    len(thread_records)
)

print("\nSample thread paths:")

for record in thread_records[:10]:
    print(
        record["conversation_id"],
        "→",
        " → ".join(record["thread_path"])
    )

Branch-aware threads created: 2257

Sample thread paths:
100245 → 100245 → 100244 → 100243 → 100241 → 100242
100540 → 100540 → 100539
101228 → 101228 → 101227
101231 → 101231 → 101230
101234 → 101234 → 101232 → 101233 → 101235 → 101236 → 101237
101239 → 101239 → 101238
101241 → 101241 → 101240
101244 → 101244 → 101243
101246 → 101246 → 101245
101249 → 101249 → 101248 → 101247


In [49]:
# ============================================
# Cell 42 — Validate branch-aware threads
# ============================================

print("Total branch-aware threads:", len(thread_records))

# Check for empty threads
empty_threads = [
    record
    for record in thread_records
    if not record["thread_path"]
]

print("Empty threads:", len(empty_threads))

# Check for duplicate tweet IDs inside a thread
duplicate_threads = []

for record in thread_records:
    path = record["thread_path"]

    if len(path) != len(set(path)):
        duplicate_threads.append(record)

print("Threads containing duplicate tweet IDs:", len(duplicate_threads))

# Thread length statistics
thread_lengths = [
    len(record["thread_path"])
    for record in thread_records
]

print("\nThread length statistics:")
print("Shortest thread:", min(thread_lengths))
print("Longest thread:", max(thread_lengths))
print(
    "Average thread length:",
    round(sum(thread_lengths) / len(thread_lengths), 2)
)

Total branch-aware threads: 2257
Empty threads: 0
Threads containing duplicate tweet IDs: 0

Thread length statistics:
Shortest thread: 2
Longest thread: 12
Average thread length: 3.25


In [51]:
# ============================================
# Cell 43 — Check tweet coverage
# ============================================

all_thread_tweet_ids = set()

for record in thread_records:
    all_thread_tweet_ids.update(record["thread_path"])

conversation_tweet_ids = set(
    conversation_df["tweet_id"]
)

missing_tweets = conversation_tweet_ids - all_thread_tweet_ids
extra_tweets = all_thread_tweet_ids - conversation_tweet_ids

print("Tweets in conversation_df:", len(conversation_tweet_ids))
print("Tweets appearing in threads:", len(all_thread_tweet_ids))

print("\nMissing tweets:", len(missing_tweets))
print("Extra tweets:", len(extra_tweets))

if missing_tweets:
    print("\nSample missing tweets:")
    print(list(missing_tweets)[:20])

Tweets in conversation_df: 6836
Tweets appearing in threads: 6836

Missing tweets: 0
Extra tweets: 0


In [52]:
# ============================================
# Cell 45 — Create tweet lookup
# ============================================

tweet_lookup = (
    conversation_df
    .set_index("tweet_id")
    .to_dict("index")
)

print("Tweet lookup created:", len(tweet_lookup))

Tweet lookup created: 6836


In [55]:
# ============================================
# Cell 46 — Convert thread paths to messages
# ============================================

conversation_threads = []

for record in thread_records:

    messages = []

    for tweet_id in record["thread_path"]:

        tweet = tweet_lookup.get(tweet_id)

        if tweet is None:
            continue

        messages.append({
            "tweet_id": tweet_id,
            "author_id": tweet["author_id"],
            "role": tweet["role"],
            "text": tweet["text_raw"],
            "text_clean": tweet["text_clean"],
            "created_at": tweet["created_at"]
        })

    conversation_threads.append({
        "conversation_id": record["conversation_id"],
        "thread_path": record["thread_path"],
        "messages": messages
    })

print(
    "Conversation threads created:",
    len(conversation_threads)
)

Conversation threads created: 2257


In [56]:
# ============================================
# Cell 47 — Validate customer/support roles
# ============================================

customer_messages = 0
support_messages = 0
other_messages = 0

for thread in conversation_threads:

    for message in thread["messages"]:

        if message["role"] == "customer":
            customer_messages += 1

        elif message["role"] == "support":
            support_messages += 1

        else:
            other_messages += 1

print("Customer messages:", customer_messages)
print("Support messages:", support_messages)
print("Other messages:", other_messages)

print(
    "Total messages:",
    customer_messages
    + support_messages
    + other_messages
)

Customer messages: 4018
Support messages: 3309
Other messages: 0
Total messages: 7327


In [57]:
# ============================================
# Cell 48 — Inspect real conversations
# ============================================

for thread in conversation_threads[:10]:

    print("\n" + "=" * 80)
    print("Conversation ID:", thread["conversation_id"])

    for message in thread["messages"]:

        print(f"\n{message['role'].upper()}:")
        print(message["text"])


Conversation ID: 100245

CUSTOMER:
What is this @AppleSupport? since I upgraded with this $hi&amp;*y #macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!

SUPPORT:
@137967 We'd like to help. Does this seem to persist when using multiple apps? Have you tried rebooting to see if that helps? Send us a DM, we'll be glad to work on this with you. https://t.co/GDrqU22YpT

CUSTOMER:
@AppleSupport I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.

SUPPORT:
@137967 Either way, let us know what happens via DM. We want to make sure this is fully resolved. https://t.co/GDrqU22YpT

CUSTOMER:
@AppleSupport for now I just restarted, it doesn't lag yet. I will let you know when it happen again.

Conversation ID: 100540

CUSTOMER:
My iPhone been moving slow af the past couple weeks. I need answers @AppleSuppo

In [76]:
# ============================================
# Cell 49 — Extract valid customer/support pairs
# ============================================

support_pairs = []

for thread in conversation_threads:

    messages = thread["messages"]

    for i in range(len(messages) - 1):

        customer = messages[i]
        support = messages[i + 1]

        # We only want customer → support interactions
        if (
            customer["role"] == "customer"
            and support["role"] == "support"
        ):

            # -----------------------------
            # Get customer message
            # -----------------------------
            customer_text = customer["text_clean"]

            # If cleaned text is missing,
            # use the original/raw text
            if pd.isna(customer_text):
                customer_text = customer["text"]

            # -----------------------------
            # Get support response
            # -----------------------------
            # Support text_clean is currently empty,
            # so use the raw text stored in "text"
            support_text = support["text"]

            # -----------------------------
            # Skip missing values
            # -----------------------------
            if pd.isna(customer_text) or pd.isna(support_text):
                continue

            customer_text = str(customer_text).strip()
            support_text = str(support_text).strip()

            # Skip empty messages
            if not customer_text or not support_text:
                continue

            # -----------------------------
            # Store valid pair
            # -----------------------------
            support_pairs.append({
                "conversation_id": thread["conversation_id"],
                "customer_tweet_id": customer["tweet_id"],
                "support_tweet_id": support["tweet_id"],
                "customer_message": customer_text,
                "support_response": support_text
            })


print(
    "Valid customer → support pairs:",
    len(support_pairs)
)

Valid customer → support pairs: 3276


In [77]:
# ============================================
# Cell 50 — Inspect valid customer/support pairs
# ============================================

for i, pair in enumerate(support_pairs[:20], start=1):

    print("\n" + "=" * 80)
    print(f"Example {i}")
    print("Conversation:", pair["conversation_id"])

    print("\nCUSTOMER:")
    print(pair["customer_message"])

    print("\nSUPPORT:")
    print(pair["support_response"])


Example 1
Conversation: 100245

CUSTOMER:
What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!

SUPPORT:
@137967 We'd like to help. Does this seem to persist when using multiple apps? Have you tried rebooting to see if that helps? Send us a DM, we'll be glad to work on this with you. https://t.co/GDrqU22YpT

Example 2
Conversation: 100245

CUSTOMER:
I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.

SUPPORT:
@137967 Either way, let us know what happens via DM. We want to make sure this is fully resolved. https://t.co/GDrqU22YpT

Example 3
Conversation: 100540

CUSTOMER:
My iPhone been moving slow af the past couple weeks. I need answers

SUPPORT:
@137986 We’d love to help with the performance of your iPhone. To start, please send us a DM confirm

In [78]:
# ============================================
# Cell 51 — Check duplicate customer messages
# ============================================

customer_texts = [
    pair["customer_message"].strip().lower()
    for pair in support_pairs
]

unique_customer_texts = set(customer_texts)

print("Total valid pairs:", len(support_pairs))
print("Unique customer messages:", len(unique_customer_texts))
print(
    "Duplicate customer messages:",
    len(support_pairs) - len(unique_customer_texts)
)

Total valid pairs: 3276
Unique customer messages: 3023
Duplicate customer messages: 253


In [79]:
# ============================================
# Cell 52 — Check exact duplicate customer/support pairs
# ============================================

pair_keys = [
    (
        pair["customer_message"].strip().lower(),
        pair["support_response"].strip().lower()
    )
    for pair in support_pairs
]

unique_pair_keys = set(pair_keys)

print("Total valid pairs:", len(support_pairs))
print("Unique customer-support pairs:", len(unique_pair_keys))
print(
    "Exact duplicate pairs:",
    len(support_pairs) - len(unique_pair_keys)
)

Total valid pairs: 3276
Unique customer-support pairs: 3092
Exact duplicate pairs: 184


In [80]:
# ============================================
# Cell 53 — Inspect exact duplicate pairs
# ============================================

from collections import Counter

pair_counts = Counter(pair_keys)

duplicate_keys = [
    key
    for key, count in pair_counts.items()
    if count > 1
]

print(
    "Number of duplicated pair types:",
    len(duplicate_keys)
)

for i, key in enumerate(duplicate_keys[:10], start=1):

    customer_text, support_text = key

    print("\n" + "=" * 80)
    print("Duplicate example", i)

    print("\nCUSTOMER:")
    print(customer_text)

    print("\nSUPPORT:")
    print(support_text)

    print("\nCount:", pair_counts[key])

Number of duplicated pair types: 145

Duplicate example 1

CUSTOMER:
applesupport since sierra update (10.12.6) mouse & keyboard constantly losing/regaining bluetooth esp in fcpx. bought wired set up & even then mac freezes on occasions. out of pocket by €300 with this upgrade and doesn't function properly - any suggestions

SUPPORT:
@139271 thanks for reaching out. we know how important it is to have your mac working as expected. we'd like some details to assist. are these issues currently occurring on 10.12.6 or have you updated to high sierra? also, let us know the mac model you're using.

Count: 2

Duplicate example 2

CUSTOMER:
thanks for reply. these issues occurring on 10.12.6 since upgrading from el capitan. (i didn't experience these problems in el cap) nervous about updating to high sierra (i had to replace an unsupported audio interface on last upgrade) i have a mid 2011 imac 21.5 inch

SUPPORT:
@139271 our pleasure. since the issues began are there any steps you've attempte

In [81]:
# ============================================
# Cell 54 — Remove exact duplicate pairs
# ============================================

clean_support_pairs = []
seen_pairs = set()

for pair in support_pairs:

    # Normalize both sides only for duplicate detection
    customer_key = pair["customer_message"].strip().lower()
    support_key = pair["support_response"].strip().lower()

    pair_key = (
        customer_key,
        support_key
    )

    # Keep only the first occurrence
    if pair_key not in seen_pairs:

        seen_pairs.add(pair_key)
        clean_support_pairs.append(pair)

print("Original pairs:", len(support_pairs))
print("Clean pairs:", len(clean_support_pairs))
print(
    "Removed exact duplicates:",
    len(support_pairs) - len(clean_support_pairs)
)

Original pairs: 3276
Clean pairs: 3092
Removed exact duplicates: 184


In [82]:
# ============================================
# Cell 56 — Analyze Apple Support response patterns
# ============================================

import re
from collections import Counter

response_patterns = {
    "dm_request": 0,
    "question": 0,
    "troubleshooting": 0,
    "link": 0,
    "acknowledgement": 0
}

for pair in clean_support_pairs:

    response = pair["support_response"].lower()

    # DM / private support
    if (
        "dm" in response
        or "direct message" in response
        or "send us a message" in response
    ):
        response_patterns["dm_request"] += 1

    # Asking for information
    if "?" in response:
        response_patterns["question"] += 1

    # Troubleshooting / action-oriented response
    troubleshooting_words = [
        "restart",
        "reboot",
        "update",
        "backup",
        "check",
        "settings",
        "reset",
        "install",
        "remove",
        "try",
        "test"
    ]

    if any(
        word in response
        for word in troubleshooting_words
    ):
        response_patterns["troubleshooting"] += 1

    # Contains external link
    if "http://" in response or "https://" in response:
        response_patterns["link"] += 1

    # Basic acknowledgement
    acknowledgement_words = [
        "thanks for reaching out",
        "thank you for reaching out",
        "we'd like to help",
        "we'd be happy to help",
        "we can help",
        "we'd love to help",
        "we're here to help"
    ]

    if any(
        phrase in response
        for phrase in acknowledgement_words
    ):
        response_patterns["acknowledgement"] += 1


print("Apple Support Response Pattern Analysis")
print("=" * 50)

for pattern, count in response_patterns.items():

    percentage = (
        count / len(clean_support_pairs)
    ) * 100

    print(
        f"{pattern:20s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Apple Support Response Pattern Analysis
dm_request          : 1526 (49.35%)
question            : 1144 (37.00%)
troubleshooting     : 1052 (34.02%)
link                : 2063 (66.72%)
acknowledgement     :  629 (20.34%)


In [83]:
# ============================================
# Cell 57 — Classify primary Apple response behavior
# ============================================

def classify_response(response):

    response = response.lower().strip()

    has_dm = (
        "dm" in response
        or "direct message" in response
        or "send us a message" in response
        or "message us" in response
    )

    has_question = "?" in response

    troubleshooting_words = [
        "restart",
        "reboot",
        "update",
        "backup",
        "check",
        "settings",
        "reset",
        "install",
        "remove",
        "try",
        "test",
        "turn off",
        "turn on"
    ]

    has_troubleshooting = any(
        word in response
        for word in troubleshooting_words
    )

    has_link = (
        "http://" in response
        or "https://" in response
    )

    # Primary behavior
    # More specific actions/questions take priority
    if has_troubleshooting:
        return "troubleshooting"

    elif has_question:
        return "information_request"

    elif has_dm:
        return "dm_escalation"

    elif has_link:
        return "link_resource"

    elif any(
        phrase in response
        for phrase in [
            "thanks for reaching out",
            "thank you for reaching out",
            "we'd like to help",
            "we'd be happy to help",
            "we can help",
            "we'd love to help",
            "we're here to help"
        ]
    ):
        return "acknowledgement"

    else:
        return "other"


# Apply classification
for pair in clean_support_pairs:

    pair["response_behavior"] = classify_response(
        pair["support_response"]
    )


# Count behaviors
behavior_counts = Counter(
    pair["response_behavior"]
    for pair in clean_support_pairs
)


print("Primary Apple Support Response Behavior")
print("=" * 50)

for behavior, count in behavior_counts.most_common():

    percentage = (
        count / len(clean_support_pairs)
    ) * 100

    print(
        f"{behavior:22s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Primary Apple Support Response Behavior
troubleshooting       : 1054 (34.09%)
dm_escalation         :  908 (29.37%)
information_request   :  626 (20.25%)
link_resource         :  337 (10.90%)
other                 :  142 (4.59%)
acknowledgement       :   25 (0.81%)


In [85]:
# ============================================
# Cell 58 — Inspect DM escalation responses
# ============================================

dm_responses = [
    pair
    for pair in clean_support_pairs
    if pair["response_behavior"] == "dm_escalation"
]

print(
    "Total DM escalation responses:",
    len(dm_responses)
)

for i, pair in enumerate(dm_responses[:20], start=1):

    print("\n" + "=" * 80)
    print(f"Example {i}")
    print("Conversation:", pair["conversation_id"])

    print("\nCUSTOMER:")
    print(pair["customer_message"])

    print("\nAPPLE SUPPORT:")
    print(pair["support_response"])

Total DM escalation responses: 908

Example 1
Conversation: 100245

CUSTOMER:
I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.

APPLE SUPPORT:
@137967 Either way, let us know what happens via DM. We want to make sure this is fully resolved. https://t.co/GDrqU22YpT

Example 2
Conversation: 101241

CUSTOMER:
anyone else’s phone changing “it” to I.T

APPLE SUPPORT:
@138155 We've received your DM and will respond to you there shortly.

Example 3
Conversation: 101244

CUSTOMER:
Spam emails ....

APPLE SUPPORT:
@138156 Thank you for reaching out to us and providing us with this information. Please report this phishing message using the info here: https://t.co/LNMCdqt6fD

Example 4
Conversation: 101249

CUSTOMER:
@115858 @116333 My Apple account’s Family Sharing has been broken for 10 weeks now, and support doesn’t seem any closer to fixing this. When asked for alternative solutions the answer is to have yet more patience. This is ho

In [86]:
# ============================================
# Cell 59 — Analyze quality of DM responses
# ============================================

def has_actionable_guidance(response):

    response = response.lower()

    actionable_words = [
        "restart",
        "reboot",
        "update",
        "backup",
        "check",
        "settings",
        "reset",
        "install",
        "remove",
        "try",
        "test",
        "confirm",
        "tell us",
        "let us know",
        "which",
        "version",
        "model",
        "serial",
        "case number",
        "country",
        "purchase",
        "error"
    ]

    return any(
        word in response
        for word in actionable_words
    )


weak_dm = 0
useful_dm = 0

for pair in clean_support_pairs:

    response = pair["support_response"].lower()

    has_dm = (
        "dm" in response
        or "direct message" in response
        or "send us a message" in response
        or "message us" in response
    )

    if not has_dm:
        continue

    if has_actionable_guidance(response):
        useful_dm += 1
    else:
        weak_dm += 1


total_dm = useful_dm + weak_dm

print("DM response analysis")
print("=" * 50)

print("Total DM responses:", total_dm)
print(
    "DM + useful guidance:",
    useful_dm,
    f"({useful_dm / total_dm * 100:.2f}%)"
)
print(
    "Weak / generic DM responses:",
    weak_dm,
    f"({weak_dm / total_dm * 100:.2f}%)"
)

DM response analysis
Total DM responses: 1528
DM + useful guidance: 793 (51.90%)
Weak / generic DM responses: 735 (48.10%)


In [87]:
# ============================================
# Cell 60 — Inspect weak / generic DM responses
# ============================================

weak_dm_responses = []

for pair in clean_support_pairs:

    response = pair["support_response"].lower()

    has_dm = (
        "dm" in response
        or "direct message" in response
        or "send us a message" in response
        or "message us" in response
    )

    if not has_dm:
        continue

    if not has_actionable_guidance(response):
        weak_dm_responses.append(pair)


print("Weak / generic DM responses:", len(weak_dm_responses))

for i, pair in enumerate(weak_dm_responses[:30], start=1):

    print("\n" + "=" * 80)
    print(f"Example {i}")
    print("Conversation:", pair["conversation_id"])

    print("\nCUSTOMER:")
    print(pair["customer_message"])

    print("\nAPPLE SUPPORT:")
    print(pair["support_response"])

Weak / generic DM responses: 735

Example 1
Conversation: 101241

CUSTOMER:
anyone else’s phone changing “it” to I.T

APPLE SUPPORT:
@138155 We've received your DM and will respond to you there shortly.

Example 2
Conversation: 101244

CUSTOMER:
Spam emails ....

APPLE SUPPORT:
@138156 Thank you for reaching out to us and providing us with this information. Please report this phishing message using the info here: https://t.co/LNMCdqt6fD

Example 3
Conversation: 101253

CUSTOMER:
please, since the last update 10.13.1, there is a huge delay in the refresh of the files in the Finder. Same problem was before but it was fixed in the previous version. Now, the problem appears again

APPLE SUPPORT:
@138160 We want to ensure we're able to get your Mac running as expected. Please meet us in DM with more information on the exact issue you're experiencing. 

You can meet us in DM, here: https://t.co/GDrqU22YpT

Example 4
Conversation: 101304

CUSTOMER:
how do I go about raising a high level compl

In [88]:
# ============================================
# Cell 61 — Final response quality categories
# ============================================

def classify_response_quality(response):

    response_lower = response.lower().strip()

    has_dm = (
        "dm" in response_lower
        or "direct message" in response_lower
        or "send us a message" in response_lower
        or "message us" in response_lower
    )

    has_link = (
        "http://" in response_lower
        or "https://" in response_lower
    )

    troubleshooting_words = [
        "restart",
        "reboot",
        "update",
        "backup",
        "reset",
        "install",
        "remove",
        "check",
        "settings",
        "turn off",
        "turn on",
        "test",
        "try",
        "report"
    ]

    information_requests = [
        "what",
        "which",
        "how",
        "when",
        "where",
        "version",
        "model",
        "serial",
        "case number",
        "details",
        "error",
        "purchased",
        "software"
    ]

    has_troubleshooting = any(
        word in response_lower
        for word in troubleshooting_words
    )

    has_information_request = (
        "?" in response_lower
        or any(
            phrase in response_lower
            for phrase in information_requests
        )
    )

    # 1. Concrete help/resource
    if has_troubleshooting or has_link:
        return "actionable"

    # 2. DM escalation with specific information request
    if has_dm and has_information_request:
        return "guided_escalation"

    # 3. Generic DM / generic acknowledgement
    if has_dm:
        return "generic_escalation"

    # 4. Other responses
    return "other"


# Apply classification
for pair in clean_support_pairs:
    pair["response_quality"] = classify_response_quality(
        pair["support_response"]
    )


# Count categories
quality_counts = Counter(
    pair["response_quality"]
    for pair in clean_support_pairs
)

print("Response quality distribution")
print("=" * 50)

for quality, count in quality_counts.most_common():

    percentage = (
        count / len(clean_support_pairs)
    ) * 100

    print(
        f"{quality:22s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Response quality distribution
actionable            : 2476 (80.08%)
other                 :  562 (18.18%)
generic_escalation    :   48 (1.55%)
guided_escalation     :    6 (0.19%)


In [89]:
# ============================================
# Cell 62 — Improved response quality rule
# ============================================

def classify_response_quality(response):

    response_lower = response.lower().strip()

    has_dm = (
        "dm" in response_lower
        or "direct message" in response_lower
        or "send us a message" in response_lower
        or "message us" in response_lower
    )

    has_link = (
        "http://" in response_lower
        or "https://" in response_lower
    )

    troubleshooting_words = [
        "restart",
        "reboot",
        "update",
        "backup",
        "reset",
        "install",
        "remove",
        "check",
        "settings",
        "turn off",
        "turn on",
        "test",
        "try",
        "report",
        "use the link",
        "follow the steps"
    ]

    information_request_patterns = [
        "?",
        "what",
        "which",
        "how",
        "when",
        "where",
        "version",
        "model",
        "serial",
        "case number",
        "details",
        "error",
        "purchased",
        "software",
        "device",
        "country",
        "app",
        "tell us",
        "let us know"
    ]

    has_troubleshooting = any(
        word in response_lower
        for word in troubleshooting_words
    )

    has_information_request = any(
        phrase in response_lower
        for phrase in information_request_patterns
    )

    # ------------------------------------------------
    # 1. Guided escalation
    # DM + asks for specific information / next step
    # ------------------------------------------------

    if has_dm and has_information_request:
        return "guided_escalation"

    # ------------------------------------------------
    # 2. Generic escalation
    # DM without meaningful guidance
    # ------------------------------------------------

    if has_dm:
        return "generic_escalation"

    # ------------------------------------------------
    # 3. Actionable response
    # Troubleshooting or useful resource
    # ------------------------------------------------

    if has_troubleshooting or has_link:
        return "actionable"

    # ------------------------------------------------
    # 4. Other
    # ------------------------------------------------

    return "other"


# Apply new classification
for pair in clean_support_pairs:

    pair["response_quality"] = classify_response_quality(
        pair["support_response"]
    )


# Count categories
quality_counts = Counter(
    pair["response_quality"]
    for pair in clean_support_pairs
)


print("Improved response quality distribution")
print("=" * 55)

for quality, count in quality_counts.most_common():

    percentage = (
        count / len(clean_support_pairs)
    ) * 100

    print(
        f"{quality:22s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Improved response quality distribution
guided_escalation     : 1025 (33.15%)
actionable            : 1002 (32.41%)
other                 :  562 (18.18%)
generic_escalation    :  503 (16.27%)


In [90]:
# ============================================
# Cell 63 — Inspect "other" responses
# ============================================

other_responses = [
    pair
    for pair in clean_support_pairs
    if pair["response_quality"] == "other"
]

print("Total OTHER responses:", len(other_responses))

for i, pair in enumerate(other_responses[:30], start=1):

    print("\n" + "=" * 80)
    print(f"Example {i}")
    print("Conversation:", pair["conversation_id"])

    print("\nCUSTOMER:")
    print(pair["customer_message"])

    print("\nAPPLE SUPPORT:")
    print(pair["support_response"])

Total OTHER responses: 562

Example 1
Conversation: 101239

CUSTOMER:
@115858 Just because you're releasing new phones doesn't mean you should stop making my current iPhone stop working. I have a contract and cannot just purchase a new phone every 12 months.

APPLE SUPPORT:
@138154 If at all possible, we would love to help. Can you tell us in more detail about what you're experiencing with the current iPhone you have?

Example 2
Conversation: 101266

CUSTOMER:
And before that I hadn’t done any updates between when it worked normally and the time it froze & crashed.

APPLE SUPPORT:
@138164 Thanks.  Are you able to successfully connect to any other Bluetooth devices?

Example 3
Conversation: 101302

CUSTOMER:
common guys. I rely on your products for video production for broadcast and your update to fix your security issue is completely crashing the OS... as a life long user of your products this is unacceptable

APPLE SUPPORT:
@138176 We can definitely understand and want you to be able 

In [91]:
# ============================================
# Cell 64 — Inspect potential failure cases
# ============================================

potential_failures = [
    pair
    for pair in clean_support_pairs
    if pair["response_quality"] == "generic_escalation"
]

print("Potential failure cases:", len(potential_failures))

for i, pair in enumerate(potential_failures[:50], start=1):

    print("\n" + "=" * 80)
    print(f"Example {i}")
    print("Conversation:", pair["conversation_id"])

    print("\nCUSTOMER:")
    print(pair["customer_message"])

    print("\nAPPLE SUPPORT:")
    print(pair["support_response"])

Potential failure cases: 503

Example 1
Conversation: 101241

CUSTOMER:
anyone else’s phone changing “it” to I.T

APPLE SUPPORT:
@138155 We've received your DM and will respond to you there shortly.

Example 2
Conversation: 101244

CUSTOMER:
Spam emails ....

APPLE SUPPORT:
@138156 Thank you for reaching out to us and providing us with this information. Please report this phishing message using the info here: https://t.co/LNMCdqt6fD

Example 3
Conversation: 101253

CUSTOMER:
please, since the last update 10.13.1, there is a huge delay in the refresh of the files in the Finder. Same problem was before but it was fixed in the previous version. Now, the problem appears again

APPLE SUPPORT:
@138160 We want to ensure we're able to get your Mac running as expected. Please meet us in DM with more information on the exact issue you're experiencing. 

You can meet us in DM, here: https://t.co/GDrqU22YpT

Example 4
Conversation: 102363

CUSTOMER:
Dear , the feature that is suppose to keep Live 

In [92]:
# ============================================
# Cell 65 — Prepare customer messages
# ============================================

customer_messages = [
    pair["customer_message"]
    for pair in clean_support_pairs
]

print("Customer messages available:", len(customer_messages))
print("Unique customer messages:", len(set(
    msg.strip().lower()
    for msg in customer_messages
)))

print("\nSample customer messages:")
print("=" * 70)

for i, message in enumerate(customer_messages[:20], start=1):
    print(f"{i}. {message}")

Customer messages available: 3092
Unique customer messages: 3023

Sample customer messages:
1. What is this ? since I upgraded with this $hi&*y macOS my instant click before is now lagging to almost a minute and that disrupts my writing e-mails (work and personal) wow! this is getting crazy everyday not enjoying Apple Macbook NOW!
2. I have tried restarting my macbook but it keeps lagging. I will try rebooting and close all apps once again.
3. My iPhone been moving slow af the past couple weeks. I need answers
4. Dear god not again,
5. Noticed a bug on my @115858 iPhone X/iOS 11.1.2 While I’m on the phone I can’t close apps.
6. leg dit even uit? Dit is mijn oud e-mail adress/apple id, sinds een recente update krijg ik constant deze melding. Ik heb sinds paar jaar een nieuw e-mail adres en ook een apple id op dit adres. Ik was (nog altijd) het wachtwoord kwijt van het oude e-mail & apple-id
7. Since a recent update i’m constantly getting these notifications about my old apple ID /e-mail

In [93]:
# ============================================
# Cell 66 — Explore recurring intent keywords
# ============================================

from collections import Counter
import re

intent_keywords = {
    "performance": [
        "slow", "lag", "lagging", "freeze", "freezing",
        "crash", "crashes", "hang", "hanging"
    ],

    "battery": [
        "battery", "drain", "charging", "charge",
        "charger", "battery life"
    ],

    "update": [
        "update", "updated", "upgrading", "upgrade",
        "ios 11", "ios 10", "high sierra", "macos"
    ],

    "apple_id_icloud": [
        "apple id", "icloud", "i cloud",
        "account", "password", "sign in", "login"
    ],

    "keyboard_autocorrect": [
        "keyboard", "autocorrect", "auto correct",
        "typing", "type", "it", "i.t"
    ],

    "connectivity": [
        "wifi", "wi-fi", "bluetooth",
        "hotspot", "connect", "connection",
        "pairing", "paired"
    ],

    "audio": [
        "sound", "audio", "headphone",
        "headphones", "earphone", "earphones",
        "speaker"
    ],

    "display_touch": [
        "screen", "display", "touch",
        "brightness", "greenline"
    ],

    "apps": [
        "app", "apps", "itunes", "safari",
        "spotify", "facetime", "podcasts",
        "finder", "app store"
    ],

    "family_sharing": [
        "family sharing"
    ],

    "spam_security": [
        "spam", "phishing", "scam",
        "security", "fraud"
    ],

    "hardware": [
        "macbook", "iphone", "ipad",
        "apple watch", "airpods",
        "trackpad", "mouse", "camera"
    ]
}


keyword_counts = Counter()

for message in customer_messages:

    text = message.lower()

    for intent, keywords in intent_keywords.items():

        if any(keyword in text for keyword in keywords):
            keyword_counts[intent] += 1


print("Keyword-based intent signals")
print("=" * 60)

for intent, count in keyword_counts.most_common():

    percentage = count / len(customer_messages) * 100

    print(
        f"{intent:22s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Keyword-based intent signals
keyboard_autocorrect  : 1664 (53.82%)
update                :  832 (26.91%)
apps                  :  832 (26.91%)
hardware              :  726 (23.48%)
performance           :  313 (10.12%)
battery               :  267 (8.64%)
display_touch         :  203 (6.57%)
connectivity          :  163 (5.27%)
apple_id_icloud       :  153 (4.95%)
audio                 :   62 (2.01%)
spam_security         :   34 (1.10%)
family_sharing        :    1 (0.03%)


In [94]:
# ============================================
# Cell 67 — Clean Intent Signal Analysis
# ============================================

intent_keywords_clean = {
    "performance": [
        "slow", "lag", "lagging", "freeze",
        "freezing", "crash", "crashes",
        "crashed", "hang", "hanging"
    ],

    "battery": [
        "battery", "battery life", "battery drain",
        "drain", "charging", "charger"
    ],

    "software_update": [
        "update", "updated", "upgrading",
        "upgrade", "ios 11", "ios 10",
        "ios 11.1", "macos", "high sierra"
    ],

    "apple_id_icloud": [
        "apple id", "icloud", "i cloud",
        "sign in", "login", "password",
        "forgot password"
    ],

    "keyboard_autocorrect": [
        "autocorrect", "auto correct",
        "keyboard", "typing", "typing issue"
    ],

    "connectivity": [
        "wifi", "wi-fi", "bluetooth",
        "hotspot", "pairing", "paired",
        "cannot connect", "can't connect",
        "connection"
    ],

    "audio": [
        "sound", "audio", "headphone",
        "headphones", "earphone",
        "earphones", "speaker"
    ],

    "display_touch": [
        "screen", "display", "touch",
        "brightness", "touchscreen"
    ],

    "apps_services": [
        "app store", "itunes", "safari",
        "facetime", "finder", "icloud drive",
        "photos app", "music app"
    ],

    "family_sharing": [
        "family sharing"
    ],

    "security_spam": [
        "spam email", "spam emails",
        "phishing", "scam", "fraud"
    ],

    "file_sharing": [
        "file sharing", "file sharing issue",
        "finder", "files"
    ]
}


keyword_counts_clean = Counter()

for message in customer_messages:

    text = message.lower()

    for intent, keywords in intent_keywords_clean.items():

        matched = False

        for keyword in keywords:

            # Exact phrase/word matching
            if " " in keyword:
                if keyword in text:
                    matched = True
                    break
            else:
                if re.search(r"\b" + re.escape(keyword) + r"\b", text):
                    matched = True
                    break

        if matched:
            keyword_counts_clean[intent] += 1


print("Cleaned keyword-based intent signals")
print("=" * 65)

for intent, count in keyword_counts_clean.most_common():

    percentage = count / len(customer_messages) * 100

    print(
        f"{intent:22s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Cleaned keyword-based intent signals
software_update       :  783 (25.32%)
battery               :  229 (7.41%)
display_touch         :  170 (5.50%)
performance           :  161 (5.21%)
apps_services         :  150 (4.85%)
apple_id_icloud       :  125 (4.04%)
connectivity          :  118 (3.82%)
keyboard_autocorrect  :   82 (2.65%)
audio                 :   59 (1.91%)
security_spam         :   10 (0.32%)
file_sharing          :    8 (0.26%)
family_sharing        :    1 (0.03%)


In [95]:
# ============================================
# Cell 68 — Initial Intent Assignment
# ============================================

def assign_initial_intent(text):
    text = text.lower()

    # 1. Account / iCloud
    if any(keyword in text for keyword in [
        "apple id",
        "icloud",
        "i cloud",
        "sign in",
        "login",
        "forgot password",
        "password"
    ]):
        return "account_icloud"

    # 2. Connectivity
    if any(keyword in text for keyword in [
        "wifi",
        "wi-fi",
        "bluetooth",
        "hotspot",
        "pairing",
        "paired",
        "cannot connect",
        "can't connect"
    ]):
        return "connectivity"

    # 3. Battery / power
    if any(keyword in text for keyword in [
        "battery",
        "battery life",
        "battery drain",
        "drain",
        "charging",
        "charger"
    ]):
        return "battery_power"

    # 4. Software / update
    if any(keyword in text for keyword in [
        "update",
        "updated",
        "upgrading",
        "upgrade",
        "ios 11",
        "ios 10",
        "macos",
        "high sierra"
    ]):
        return "software_update"

    # 5. Performance
    if any(keyword in text for keyword in [
        "slow",
        "lag",
        "lagging",
        "freeze",
        "freezing",
        "crash",
        "crashes",
        "crashed",
        "hang",
        "hanging"
    ]):
        return "performance"

    # 6. Apps / Apple services
    if any(keyword in text for keyword in [
        "app store",
        "itunes",
        "safari",
        "facetime",
        "finder",
        "photos app",
        "music app"
    ]):
        return "apps_services"

    # 7. Device hardware
    if any(keyword in text for keyword in [
        "screen",
        "display",
        "touch",
        "touchscreen",
        "brightness",
        "sound",
        "audio",
        "headphone",
        "headphones",
        "earphone",
        "earphones",
        "speaker"
    ]):
        return "device_hardware"

    # 8. Everything else
    return "other_support"


initial_intents = [
    assign_initial_intent(message)
    for message in customer_messages
]

intent_distribution = Counter(initial_intents)

print("Initial intent distribution")
print("=" * 65)

for intent, count in intent_distribution.most_common():

    percentage = count / len(initial_intents) * 100

    print(
        f"{intent:20s}: "
        f"{count:4d} "
        f"({percentage:.2f}%)"
    )

Initial intent distribution
other_support       : 1630 (52.72%)
software_update     :  653 (21.12%)
battery_power       :  224 (7.24%)
performance         :  147 (4.75%)
device_hardware     :  130 (4.20%)
account_icloud      :  127 (4.11%)
connectivity        :  110 (3.56%)
apps_services       :   71 (2.30%)


In [96]:
# ============================================
# Cell 69 — Inspect Initial "Other" Messages
# ============================================

other_messages = [
    message
    for message, intent in zip(customer_messages, initial_intents)
    if intent == "other_support"
]

print("Messages classified as other_support:", len(other_messages))
print("=" * 80)

for i, message in enumerate(other_messages[:100], start=1):
    print(f"{i}. {message}")
    print("-" * 80)

Messages classified as other_support: 1630
1. Dear god not again,
--------------------------------------------------------------------------------
2. Yes ofcourse, iphone 6s version 11.1.2 (15B202)
--------------------------------------------------------------------------------
3. @115858 Just because you're releasing new phones doesn't mean you should stop making my current iPhone stop working. I have a contract and cannot just purchase a new phone every 12 months.
--------------------------------------------------------------------------------
4. Spam emails ....
--------------------------------------------------------------------------------
5. @115858 @116333 My Apple account’s Family Sharing has been broken for 10 weeks now, and support doesn’t seem any closer to fixing this. When asked for alternative solutions the answer is to have yet more patience. This is horrible :( https://t.co/1W9S7hFGwC
--------------------------------------------------------------------------------
6. wh

In [97]:
# Cell 47 — Save reconstructed conversation dataset

import os

os.makedirs("../data/processed", exist_ok=True)

conversation_df.to_csv(
    "../data/processed/conversation_df.csv",
    index=False
)

print("Saved conversation dataset successfully.")
print("Rows:", len(conversation_df))
print("Columns:", len(conversation_df.columns))
print("File: ../data/processed/conversation_df.csv")

Saved conversation dataset successfully.
Rows: 6836
Columns: 16
File: ../data/processed/conversation_df.csv


In [98]:
# Cell 56 — Save cleaned customer-support pairs

support_pairs_df = pd.DataFrame(clean_support_pairs)

support_pairs_df.to_csv(
    "../data/processed/support_pairs.csv",
    index=False
)

print("Saved support pairs successfully.")
print("Rows:", len(support_pairs_df))
print("Columns:", len(support_pairs_df.columns))
print("File: ../data/processed/support_pairs.csv")

Saved support pairs successfully.
Rows: 3092
Columns: 7
File: ../data/processed/support_pairs.csv
